In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScale
import matplotlib.pyplot as plt
from elasticsearch import Elasticsearch

In [ ]:
# Загрузка данных
es = Elasticsearch(['http://elasticsearch:9200'])
res = es.search(
    index='logs-*',
    body={
        "query": {"match_all": {}},
        "size": 10000,
        "sort": [{"@timestamp": "desc"}]
    }
)

# Подготовка данных
logs = []
for hit in res['hits']['hits']:
    logs.append(hit['_source'])

df = pd.json_normalize(logs)

# Извлечение признаков
features = []

# Временные признаки
df['timestamp'] = pd.to_datetime(df['@timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Размер сообщения
df['timestamp'] = pd.to_datetime(df['@timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Размер сообщения
df['msg_length'] = df['message'].apply(len) if 'message' in df.columns else 0

# Статус код
if 'response.status' in df.columns:
    df['is_error'] = df['response.status'] >= 400
else:
    df['is_error'] = 0

# Выбор признаков для модели
feature_cols = ['hour', 'day_of_week', 'msg_length', 'is_error']
X = df[feature_cols].fillna(0)

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Isolation Forest
model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = model.fit_predict(X_scaled)

In [ ]:
# Визуализация
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
anomalies = df[df['anomaly'] == -1]
normals = df[df['anomaly'] == 1]

plt.scatter(normals.index, normals['msg_length'], c='blue', alpha=0.5, label='Normal')
plt.scatter(anomalies.index, anomalies['msg_length'], c='red', alpha=0.7, label='Anomaly')
plt.xlabel('Log index')
plt.ylabel('Message length')
plt.legend()
plt.title('Anomaly Detection in Logs')

plt.subplot(1, 2, 2)
df['anomaly'].value_counts().plot(kind='pie', autopct='%1.1f%%',
                                  labels=['Normal', 'Anomaly'],
                                  colors=['blue', 'red']
)
plt.title('Anomaly Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Сохранение модели
import joblib
joblib.dump(model, '/home/jovyan/work/anomaly_model.pkl')
joblib.dump(scaler, '/home/jovyan/work/scaler.pkl')
print("Model saved!")